## << Setting >>

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    GradientBoostingClassifier,
    AdaBoostClassifier
)

from sklearn.metrics import accuracy_score

In [1]:
# ============================================================
# Dataset Total Structure and Info
# ============================================================

def inspect_dataset(data, sample_size=10, descr_size=300):
    """
    다양한 형태의 데이터 구조를 범용적으로 확인한다.

    지원:
    - pandas DataFrame / Series
    - dict / sklearn Bunch
    - numpy ndarray
    - list / tuple
    - scalar
    """

    rows = []

    # ========================================================
    # DataFrame
    # ========================================================

    if isinstance(data, pd.DataFrame):

        for column in data.columns:

            value = data[column]

            rows.append({
                "Branch": column,
                "DType": str(value.dtype),
                "Struc.": f"{data.shape}",
                "Count": len(value),
                "Missing": int(value.isna().sum()),
                "Null": bool(value.isna().any()),
                "Unique cnt": int(value.nunique(dropna=True)),
                "Distribution": value.value_counts(
                    dropna=False
                ).head(sample_size).to_dict(),
                "Sample": value.head(sample_size).tolist()
            })

        return pd.DataFrame(rows)

    # ========================================================
    # Series
    # ========================================================

    if isinstance(data, pd.Series):

        value = data

        rows.append({
            "Branch": value.name or "Series",
            "DType": str(value.dtype),
            "Struc.": f"{value.shape}",
            "Count": len(value),
            "Missing": int(value.isna().sum()),
            "Null": bool(value.isna().any()),
            "Unique cnt": int(value.nunique(dropna=True)),
            "Distribution": value.value_counts(
                dropna=False
            ).head(sample_size).to_dict(),
            "Sample": value.head(sample_size).tolist()
        })

        return pd.DataFrame(rows)

    # ========================================================
    # dict / sklearn Bunch
    # ========================================================

    if isinstance(data, dict):
        items = data.items()

    elif hasattr(data, "keys"):
        items = (
            (key, data[key])
            for key in data.keys()
        )

    else:
        items = [("DATA", data)]

    # ========================================================
    # Object Inspection
    # ========================================================

    for key, value in items:

        data_type = type(value).__name__

        structure = "-"
        count = 0
        missing = 0
        null_status = False
        unique_count = "-"
        distribution = "-"
        sample = "-"

        # ----------------------------------------------------
        # DataFrame
        # ----------------------------------------------------

        if isinstance(value, pd.DataFrame):

            structure = f"{value.shape}"
            count = len(value)
            missing = int(value.isna().sum().sum())
            null_status = missing > 0
            sample = value.head(sample_size)

        # ----------------------------------------------------
        # Series
        # ----------------------------------------------------

        elif isinstance(value, pd.Series):

            structure = f"{value.shape}"
            count = len(value)
            missing = int(value.isna().sum())
            null_status = missing > 0

            unique_count = int(
                value.nunique(dropna=True)
            )

            distribution = value.value_counts(
                dropna=False
            ).head(sample_size).to_dict()

            sample = value.head(sample_size).tolist()

        # ----------------------------------------------------
        # ndarray
        # ----------------------------------------------------

        elif isinstance(value, np.ndarray):

            structure = f"{value.shape}"

            count = (
                value.shape[0]
                if value.ndim > 0
                else 1
            )

            missing = int(pd.isna(value).sum())
            null_status = missing > 0

            sample = (
                repr(value[:sample_size])
                if value.ndim > 0
                else repr(value)
            )

            if value.ndim == 1:

                try:

                    unique, counts = np.unique(
                        value,
                        return_counts=True
                    )

                    unique_count = len(unique)

                    distribution = ", ".join(
                        f"{u}:{c}"
                        for u, c in zip(unique, counts)
                    )

                except TypeError:

                    unique_count = "-"
                    distribution = "-"

            elif value.ndim == 2:

                if np.issubdtype(value.dtype, np.number):

                    distribution = (
                        f"min={value.min():.2f}, "
                        f"max={value.max():.2f}, "
                        f"mean={value.mean():.2f}"
                    )

        # ----------------------------------------------------
        # list / tuple
        # ----------------------------------------------------

        elif isinstance(value, (list, tuple)):

            structure = f"len={len(value)}"
            count = len(value)

            arr = np.array(value, dtype=object)

            missing = int(pd.isna(arr).sum())
            null_status = missing > 0

            sample = repr(value[:sample_size])

            try:

                unique = pd.unique(arr)

                unique_count = len(unique)

                distribution = ", ".join(
                    map(str, unique[:sample_size])
                )

            except Exception:

                unique_count = "-"
                distribution = "-"

        # ----------------------------------------------------
        # 문자열
        # ----------------------------------------------------

        elif isinstance(value, str):

            structure = f"len={len(value)}"
            count = 1

            sample = (
                value[:descr_size] + " ..."
                if key == "DESCR"
                else value
            )

            unique_count = 1

        # ----------------------------------------------------
        # None
        # ----------------------------------------------------

        elif value is None:

            null_status = True
            missing = 1
            sample = np.nan

        # ----------------------------------------------------
        # 기타 scalar / object
        # ----------------------------------------------------

        else:

            count = 1

            try:
                missing = int(pd.isna(value))
            except Exception:
                missing = 0

            null_status = missing > 0
            sample = repr(value)
            unique_count = 1

        # ====================================================
        # 결과 저장
        # ====================================================

        rows.append({
            "Branch": key,
            "DType": data_type,
            "Struc.": structure,
            "Count": count,
            "Missing": missing,
            "Null": null_status,
            "Unique cnt": unique_count,
            "Distribution": distribution,
            "Sample": sample
        })

    return pd.DataFrame(rows)

### Feature 리스트 

In [ ]:
## feature 별로 분류해서 처리 


#1.제외 Feature

FEATURES_EXCEPT = [
    "PassengerId",
    "Cabin",
    "Embarked",
    "SibSp",
    "Parch",
    "Ticket",
    "HasCompanion",
    # "IsFreeFare",
    # "IsChild",
    # "Age"

]


#2.DataType별 Feature  
FEATURES_CATEGORICAL = [
    "Sex",    
    "Title",
    "AgeGroup",
    # "Pclass"
    # "Embarked",
    # "CompanionGroup"
] 

FEATURES_NUMERIC = [
    "Age",    
    # "SibSp",
    # "Parch",
    "Fare",    
    "FamilySize",
]

FEATURES_ORDINAL = [
    "Pclass"
]

#3.Target 분리 (0 1 코드화 되어있음)
FEATURES_TARGET =  "Survived"  

# 4. Feature Engineering 후보
FEATURE_ENGINEERING_FEATURES = [
    "Name"
]


# 5. 현재 적용할 Feature Engineering
FEATURES_BINARY  = [
    # "HasCompanion",
    # "IsChild",
    # "IsFreeFare"
]

In [ ]:
from sklearn.model_selection import train_test_split

#1.Feature / Target 분리
x = df_modified[
    FEATURES_CATEGORICAL
    + FEATURES_NUMERIC
    + FEATURES_ORDINAL 
    + FEATURES_BINARY    
    + FEATURE_ENGINEERING_FEATURES   
]

y = df_modified[
    FEATURES_TARGET
]

#2.Train / Validation 분리 ( 8:2 ) : Target 비유은 유지하는 방향으로
x_train, x_valid, y_train, y_valid = train_test_split (
    x, y,
    test_size = 0.2,
    random_state = 42,
    stratify=y   
)

#백업
x_train_backup = x_train.copy()
x_val_backup   = x_valid.copy() 

In [ ]:

# ============================================================
# Experiment Step
# ============================================================

STEP = 3


STEP_FEATURES = {

    1: (
        FEATURES_CATEGORICAL
        + FEATURES_NUMERIC
        + FEATURES_ORDINAL
    ),

    2: (
        FEATURES_CATEGORICAL
        + FEATURES_NUMERIC
        + FEATURES_ORDINAL
        + FEATURES_BINARY
    ),

    3: (
        FEATURES_CATEGORICAL
        + [f for f in FEATURES_NUMERIC if f != "Age"]
        + FEATURES_ORDINAL
        + FEATURES_BINARY
    )
}


MODEL_FEATURES = STEP_FEATURES[STEP]


# ============================================================
# Model Feature 선택
# ============================================================

##임시


x_train_model = x_train[MODEL_FEATURES].copy()
x_valid_model = x_valid[MODEL_FEATURES].copy() 

## Pipeline ( 전처리 및 모델 적용 ) 
- 처음이라 모델별도 다 적용할 예정(기초비교자료로 사용 예정)     : AI 참조
- LogisticRegression
KNeighborsClassifier
DecisionTreeClassifier
GradientBoostingClassifier
AdaBoostClassifier

In [664]:
# ============================================================
# Preprocessor
# ============================================================

# LogisticRegression / KNN
# Numeric → StandardScaler, MinMaxScaler
# Categorical → OneHotEncoder
# Ordinal / Binary → 그대로 사용

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            # MinMaxScaler(),
            [f for f in FEATURES_NUMERIC if f in MODEL_FEATURES]
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            [f for f in FEATURES_CATEGORICAL if f in MODEL_FEATURES]
        ),
        (
            "ordinal",
            "passthrough",
            [f for f in FEATURES_ORDINAL if f in MODEL_FEATURES]
        ),
        (
            "binary",
            "passthrough",
            [f for f in FEATURES_BINARY if f in MODEL_FEATURES]
        )
    ],
    remainder="drop"
)


# Tree 계열
# Numeric → Scaling 하지 않음
# Categorical → OneHotEncoder
# Ordinal / Binary → 그대로 사용

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            [f for f in FEATURES_NUMERIC if f in MODEL_FEATURES]
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            [f for f in FEATURES_CATEGORICAL if f in MODEL_FEATURES]
        ),
        (
            "ordinal",
            "passthrough",
            [f for f in FEATURES_ORDINAL if f in MODEL_FEATURES]
        ),
        (
            "binary",
            "passthrough",
            [f for f in FEATURES_BINARY if f in MODEL_FEATURES]
        )
    ],
    remainder="drop"
)

In [665]:
# ============================================================
# Model Pipeline
# ============================================================ 

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

models = {

    # Scaling 적용
    "LogisticRegression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]),

    "KNN": Pipeline([
        ("preprocessor", preprocessor),
        ("model", KNeighborsClassifier())
    ]),

    # Scaling 미적용
    "DecisionTree": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", DecisionTreeClassifier(random_state=42))
    ]),

    "RandomForest": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestClassifier(random_state=42))
    ]),
    
    "GradientBoosting": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", GradientBoostingClassifier(random_state=42))
    ]),

    "AdaBoost": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", AdaBoostClassifier(random_state=42))
    ]),

    # "SVC": Pipeline([
    #     ("preprocessor", preprocessor),
    #     ("model", SVC()
    #     )
    # ]),

    # "XGBoost": Pipeline([
    #     ("preprocessor", preprocessor),
    #     ("model", XGBClassifier(
    #         n_estimators=100,
    #         max_depth=3,
    #         learning_rate=0.1,
    #         random_state=42,
    #         eval_metric="logloss"
    #     ))
    # ])
}

In [ ]:
# ============================================================
# Model Training & Validation( 1회 )
# ============================================================

results = {}

for name, model in models.items():

    # Train
    model.fit(
        x_train_model,
        y_train
    )

    # Validation
    y_pred = model.predict(
        x_valid_model
    )

    # Accuracy
    accuracy = accuracy_score(
        y_valid,
        y_pred
    )

    results[name] = accuracy

    print(
        f"{name:20s} : {accuracy:.4f}"
    )

LogisticRegression   : 0.8268
KNN                  : 0.7877
DecisionTree         : 0.7933
RandomForest         : 0.7933
GradientBoosting     : 0.8045
AdaBoost             : 0.8156


In [ ]:
# ============================================================
# GridSearchCV(5회)
# ============================================================
from sklearn.model_selection import GridSearchCV

param_grids = {

    "LogisticRegression": {
        "model__C": [0.01, 0.1, 1, 10, 100]
    },

    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11, 15]
    },

    "DecisionTree": {
        "model__max_depth": [2, 3, 4, 5, 6, None],
        "model__min_samples_split": [2, 5, 10]
    },

    "RandomForest": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 5, 10],
        "model__min_samples_split": [2, 5, 10],
        "model__max_features": ["sqrt", "log2"]
    },

    "GradientBoosting": {
        "model__n_estimators": [50, 100, 150, 200, 300 ],
        "model__learning_rate": [0.03, 0.05, 0.1]
    },

    "AdaBoost": {
        "model__n_estimators": [50, 100, 150],
        "model__learning_rate": [0.5, 1.0, 1.5]
    },

    # # 추가
    # "SVC": {
    #     "model__C": [0.1, 1, 10],
    #     "model__kernel": ["linear", "rbf"] 
    # }, 

    "XGBoost": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1]
    }
    
}


# ============================================================
# GridSearch 실행
# ============================================================

grid_results = {}

for name, model in models.items():

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(
        x_train_model,
        y_train
    )

    grid_results[name] = grid

    print(f"\n{name}")
    print("Best Parameter :", grid.best_params_)
    print("Best CV Score  :", f"{grid.best_score_:.4f}")


# ============================================================
# Validation 성능 확인
# ============================================================

print("\n===== Validation Accuracy =====")

grid_validation_results = {}

for name, grid in grid_results.items():

    y_pred = grid.predict(
        x_valid_model
    )

    accuracy = accuracy_score(
        y_valid,
        y_pred
    )

    grid_validation_results[name] = accuracy

    print(
        f"{name:20s} : {accuracy:.4f}"
    )


LogisticRegression
Best Parameter : {'model__C': 1}
Best CV Score  : 0.8217

KNN
Best Parameter : {'model__n_neighbors': 5}
Best CV Score  : 0.8301

DecisionTree
Best Parameter : {'model__max_depth': 3, 'model__min_samples_split': 2}
Best CV Score  : 0.8133

RandomForest
Best Parameter : {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_split': 10, 'model__n_estimators': 300}
Best CV Score  : 0.8385

GradientBoosting
Best Parameter : {'model__learning_rate': 0.1, 'model__n_estimators': 200}
Best CV Score  : 0.8273

AdaBoost
Best Parameter : {'model__learning_rate': 0.5, 'model__n_estimators': 100}
Best CV Score  : 0.8259

===== Validation Accuracy =====
LogisticRegression   : 0.8268
KNN                  : 0.7877
DecisionTree         : 0.8212
RandomForest         : 0.7821
GradientBoosting     : 0.8045
AdaBoost             : 0.8045


### 결과 History 저장

In [ ]:
# ============================================================
# Experiment History 저장
# ============================================================

CURRENT_STEP = 1
CURRENT_FEATURE = "Baseline"

experiment_history = []

PREPROCESSING_INFO = {

    "LogisticRegression": {
        "Scaling": "StandardScaler",
        "Encoding": "OneHotEncoder",
        "Imputation": "Age median"
    },

    "KNN": {
        "Scaling": "StandardScaler",
        "Encoding": "OneHotEncoder",
        "Imputation": "Age median"
    },

    "DecisionTree": {
        "Scaling": "None",
        "Encoding": "OneHotEncoder",
        "Imputation": "Age median"
    },

    "RandomForest": {
        "Scaling": "None",
        "Encoding": "OneHotEncoder",
        "Imputation": "Age median"
    },

    "GradientBoosting": {
        "Scaling": "None",
        "Encoding": "OneHotEncoder",
        "Imputation": "Age median"
    },

    "AdaBoost": {
        "Scaling": "None",
        "Encoding": "OneHotEncoder",
        "Imputation": "Age median"
    }
}


for name, grid in grid_results.items():

    preprocessing = PREPROCESSING_INFO[name]

    experiment_history.append({

        "Step": CURRENT_STEP,
        "Feature": CURRENT_FEATURE,
        "Model": name,

        "Scaling": preprocessing["Scaling"],
        "Encoding": preprocessing["Encoding"],
        "Imputation": preprocessing["Imputation"],

        "Best_Parameters": grid.best_params_,
        "CV_Score": grid.best_score_,
        "Validation_Accuracy": grid_validation_results[name]
    })


experiment_history_df = pd.DataFrame(
    experiment_history
)


print("\n===== Experiment History =====")
print(experiment_history_df)


===== Experiment History =====
   Step   Feature               Model         Scaling       Encoding  \
0     1  Baseline  LogisticRegression  StandardScaler  OneHotEncoder   
1     1  Baseline                 KNN  StandardScaler  OneHotEncoder   
2     1  Baseline        DecisionTree            None  OneHotEncoder   
3     1  Baseline    GradientBoosting            None  OneHotEncoder   
4     1  Baseline            AdaBoost            None  OneHotEncoder   

   Imputation                                    Best_Parameters  CV_Score  \
0  Age median                                  {'model__C': 0.1}  0.797833   
1  Age median                          {'model__n_neighbors': 3}  0.809032   
2  Age median  {'model__max_depth': 3, 'model__min_samples_sp...  0.814705   
3  Age median  {'model__learning_rate': 0.1, 'model__n_estima...  0.818930   
4  Age median  {'model__learning_rate': 1.5, 'model__n_estima...  0.814675   

   Validation_Accuracy  
0             0.815642  
1             0.

In [672]:
experiment_history_df

,Step,Feature,Model,Scaling,Encoding,Imputation,Best_Parameters,CV_Score,Validation_Accuracy
0,1,Baseline,LogisticRegression,StandardScaler,OneHotEncoder,Age median,{'model__C': 0.1},0.797833,0.815642
1,1,Baseline,KNN,StandardScaler,OneHotEncoder,Age median,{'model__n_neighbors': 3},0.809032,0.798883
2,1,Baseline,DecisionTree,None,OneHotEncoder,Age median,"{'model__max_depth': 3, 'model__min_samples_sp...",0.814705,0.815642
3,1,Baseline,GradientBoosting,None,OneHotEncoder,Age median,"{'model__learning_rate': 0.1, 'model__n_estima...",0.818930,0.804469
4,1,Baseline,AdaBoost,None,OneHotEncoder,Age median,"{'model__learning_rate': 1.5, 'model__n_estima...",0.814675,0.798883
5,2,+ HasCompanion,LogisticRegression,StandardScaler,OneHotEncoder,Age median,{'model__C': 1},0.802059,0.815642
6,2,+ HasCompanion,KNN,StandardScaler,OneHotEncoder,Age median,{'model__n_neighbors': 3},0.813228,0.798883
7,2,+ HasCompanion,DecisionTree,None,OneHotEncoder,Age median,"{'model__max_depth': 3, 'model__min_samples_sp...",0.814705,0.815642
8,2,+ HasCompanion,GradientBoosting,None,OneHotEncoder,Age median,"{'model__learning_rate': 0.1, 'model__n_estima...",0.818881,0.810056
9,2,+ HasCompanion,AdaBoost,None,OneHotEncoder,Age median,"{'model__learning_rate': 1.5, 'model__n_estima...",0.814675,0.798883


In [493]:
# ============================================================
# 모델 평가
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

evaluation_results = []

for name, grid in grid_results.items():

    # Validation 예측
    y_pred = grid.predict(
        x_valid_model
    )

    # Validation 확률
    y_prob = grid.predict_proba(
        x_valid_model
    )[:, 1]

    # ========================================================
    # 평가 지표
    # ========================================================

    accuracy = accuracy_score(
        y_valid,
        y_pred
    )

    precision = precision_score(
        y_valid,
        y_pred
    )

    recall = recall_score(
        y_valid,
        y_pred
    )

    f1 = f1_score(
        y_valid,
        y_pred
    )

    roc_auc = roc_auc_score(
        y_valid,
        y_prob
    )

    # Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(
        y_valid,
        y_pred
    ).ravel()

    # ========================================================
    # 결과 저장
    # ========================================================

    evaluation_results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })


# ============================================================
# Evaluation DataFrame
# ============================================================

evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,TN,FP,FN,TP
0,LogisticRegression,0.837989,0.822581,0.739130,0.778626,0.878920,99,11,18,51
1,KNN,0.782123,0.727273,0.695652,0.711111,0.836034,92,18,21,48
2,DecisionTree,0.776536,0.723077,0.681159,0.701493,0.787879,92,18,22,47
3,RandomForest,0.770950,0.705882,0.695652,0.700730,0.835771,90,20,21,48
4,GradientBoosting,0.821229,0.760563,0.782609,0.771429,0.835178,93,17,15,54
5,AdaBoost,0.815642,0.757143,0.768116,0.762590,0.854611,93,17,16,53


In [494]:
# ============================================================
# Soft Voting Ensemble
# ============================================================

from sklearn.ensemble import VotingClassifier

# 현재 GridSearch에서 최적 모델 가져오기
best_lr = grid_results["LogisticRegression"].best_estimator_
best_dt = grid_results["DecisionTree"].best_estimator_
best_gb = grid_results["GradientBoosting"].best_estimator_


# Soft Voting
voting_model = VotingClassifier(
    estimators=[
        ("lr", best_lr),
        ("dt", best_dt),
        ("gb", best_gb)
    ],
    voting="soft"
)


# 학습
voting_model.fit(
    x_train_model,
    y_train
)


# Validation 예측
y_pred = voting_model.predict(
    x_valid_model
)

# Validation 확률
y_prob = voting_model.predict_proba(
    x_valid_model
)[:, 1]

In [495]:
# ============================================================
# 앙상블 평가
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("\n===== Voting Ensemble Evaluation =====")

accuracy = accuracy_score(y_valid, y_pred)

precision = precision_score(
    y_valid,
    y_pred
)

recall = recall_score(
    y_valid,
    y_pred
)

f1 = f1_score(
    y_valid,
    y_pred
)

roc_auc = roc_auc_score(
    y_valid,
    y_prob
)

cm = confusion_matrix(
    y_valid,
    y_pred
)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print("Confusion Matrix:")
print(cm)


===== Voting Ensemble Evaluation =====
Accuracy  : 0.8212
Precision : 0.7606
Recall    : 0.7826
F1 Score  : 0.7714
ROC-AUC   : 0.8538
Confusion Matrix:
[[93 17]
 [15 54]]
